导入库

In [ ]:
import os
import random

import torch
import torch.nn as nn

from torchvision import datasets
from torchvision import transforms

from torch.utils.data import (
    DataLoader,
    random_split,
    Subset
)

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import confusion_matrix
import numpy as np

参数配置

In [ ]:
DATA_DIR = r"data/ASL_Alphabet_Dataset/asl_alphabet_train"

MODEL_PATH = "cnn_model.pth"

SAMPLE_SIZE = 50000

BATCH_SIZE = 128

EPOCHS = 10

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

os.makedirs("results", exist_ok=True)

数据预处理

In [ ]:
def get_transform():

    transform = transforms.Compose([
        transforms.Resize((64,64)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5]
        )
    ])

    return transform

获取数据集函数

In [ ]:
def get_dataset():

    dataset = datasets.ImageFolder(
        DATA_DIR,
        transform=get_transform()
    )

    return dataset

数据集信息函数

In [ ]:
def dataset_info():

    dataset = get_dataset()

    print("完整数据集:", len(dataset))

    print("类别数:", len(dataset.classes))

    print("类别名称:")

    print(dataset.classes)

获取DataLoader函数

In [ ]:
def get_dataloaders():

    full_dataset = get_dataset()

    random.seed(42)

    indices = random.sample(
        range(len(full_dataset)),
        SAMPLE_SIZE
    )

    dataset = Subset(
        full_dataset,
        indices
    )

    train_size = int(
        0.8 * len(dataset)
    )

    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size]
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4
    )

    return (
        train_loader,
        val_loader,
        full_dataset
    )

CNN模型

In [ ]:
class CNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                3,
                32,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 16 * 16,
                512
            ),

            nn.ReLU(),

            nn.Linear(
                512,
                26
            )
        )

    def forward(self, x):

        x = self.conv(x)

        x = self.fc(x)

        return x

MLP模型

In [ ]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc = nn.Sequential(
            nn.Flatten(),

            nn.Linear(64 * 64 * 3, 512),

            nn.ReLU(),

            nn.Linear(512, 26)
        )

    def forward(self, x):
        return self.fc(x)

模型相关函数

In [ ]:
def create_model():

    model = CNN().to(device)

    return model

def create_mlp():

    model = MLP().to(device)

    return model

def load_model():

    model = create_model()

    model.load_state_dict(

        torch.load(
            MODEL_PATH,
            map_location=device
        )

    )

    model.eval()

    return model



训练函数

In [ ]:
def train_model(model,model_path):

    train_loader, val_loader, _ = get_dataloaders()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=5,
        gamma=0.1
    )

    train_acc_list = []
    val_acc_list = []

    for epoch in range(EPOCHS):

        print(f"\n========== Epoch {epoch+1}/{EPOCHS} ==========")

        model.train()

        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

        train_acc = 100 * correct / total

        model.eval()

        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for images, labels in val_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)

                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)

                val_correct += (
                    predicted == labels
                ).sum().item()

        val_acc = 100 * val_correct / val_total

        train_acc_list.append(train_acc)
        val_acc_list.append(val_acc)

        scheduler.step()

        print(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"LR:{optimizer.param_groups[0]['lr']:.6f} "
            f"Train Acc:{train_acc:.2f}% "
            f"Val Acc:{val_acc:.2f}%"
        )

    torch.save(
        model.state_dict(),
        model_path
    )

    print("\n模型保存成功")

    return model,train_acc_list, val_acc_list

比较模型

In [ ]:
def compare_models(
    cnn_acc,
    mlp_acc
):

    plt.figure(figsize=(6,4))

    plt.bar(
        ["MLP", "CNN"],
        [mlp_acc, cnn_acc]
    )

    plt.ylabel("Accuracy (%)")

    plt.title("Model Comparison")

    plt.show()

准确率曲线

In [ ]:
def plot_accuracy_curve(
    train_acc_list,
    val_acc_list,
    model_name="Model"
):

    epochs_range = range(
        1,
        len(train_acc_list) + 1
    )

    plt.figure(figsize=(8,5))

    plt.plot(
        epochs_range,
        train_acc_list,
        marker="o",
        label="Train Accuracy"
    )

    plt.plot(
        epochs_range,
        val_acc_list,
        marker="s",
        label="Validation Accuracy"
    )

    plt.xlabel("Epoch")

    plt.ylabel("Accuracy (%)")

    plt.title(f"{model_name} Accuracy Curve")

    plt.legend()

    plt.grid(True)

    plt.show()

类别分布图

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

def plot_class_distribution():
    dataset = get_dataset()
    counter = Counter(dataset.targets)
    names = dataset.classes
    counts = [counter[i] for i in range(len(names))]


    plt.figure(figsize=(12, 5))
    bars = plt.bar(names, counts)

    # 在每个条形上方显示数量
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 100,
                 f'{int(height)}', ha='center', va='bottom', fontsize=8)

    plt.title("ASL Alphabet Dataset Class Distribution")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=90)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

样本展示

In [ ]:
def show_samples():

    dataset = get_dataset()

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(8,8)
    )

    for ax in axes.ravel():

        idx = random.randint(
            0,
            len(dataset)-1
        )

        image, label = dataset[idx]

        # 反归一化
        image = image * 0.5 + 0.5

        image = image.permute(
            1,
            2,
            0
        )

        ax.imshow(image)

        ax.set_title(
            dataset.classes[label]
        )

        ax.axis("off")

    plt.tight_layout()

    plt.show()

训练结果表格

In [ ]:
def create_result_table(
    train_acc_list,
    val_acc_list
):

    df = pd.DataFrame({

        "Epoch":
        range(
            1,
            len(train_acc_list)+1
        ),

        "Train Accuracy":
        train_acc_list,

        "Validation Accuracy":
        val_acc_list

    })

    return df

比较运行表

In [ ]:
def get_compare_table(cnn_val_acc, mlp_val_acc):
    import pandas as pd

    df = pd.DataFrame({
        "Epoch": list(range(1, len(cnn_val_acc)+1)),
        "CNN Val Acc": cnn_val_acc,
        "MLP Val Acc": mlp_val_acc
    })

    return df

混淆矩阵

In [ ]:
def plot_confusion_matrix():

    model = load_model()

    _, val_loader, full_dataset = get_dataloaders()

    y_true = []

    y_pred = []

    model.eval()

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)

            outputs = model(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            y_true.extend(
                labels.numpy()
            )

            y_pred.extend(
                predicted.cpu().numpy()
            )

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    classes = full_dataset.classes

    plt.figure(figsize=(12,10))

    plt.imshow(
        cm,
        interpolation="nearest",
        cmap="Blues"
    )

    plt.colorbar()

    plt.title("CNN Confusion Matrix")

    plt.xlabel("Predicted Label")

    plt.ylabel("True Label")

    plt.xticks(
        range(len(classes)),
        classes,
        rotation=45
    )

    plt.yticks(
        range(len(classes)),
        classes
    )

    plt.tight_layout()

    plt.show()

错误样本分析

In [ ]:
def show_error_samples():

    model = load_model()

    _, val_loader, full_dataset = get_dataloaders()

    wrong_images = []

    wrong_true = []

    wrong_pred = []

    model.eval()

    with torch.no_grad():

        for images, labels in val_loader:

            outputs = model(
                images.to(device)
            )

            _, predicted = torch.max(
                outputs,
                1
            )

            for i in range(len(labels)):

                if (
                    predicted[i].cpu()
                    != labels[i]
                ):

                    wrong_images.append(
                        images[i]
                    )

                    wrong_true.append(
                        labels[i].item()
                    )

                    wrong_pred.append(
                        predicted[i].item()
                    )

                if len(wrong_images) >= 9:

                    break

            if len(wrong_images) >= 9:

                break

    fig, axes = plt.subplots(
        3,
        3,
        figsize=(10,10)
    )

    for i, ax in enumerate(axes.ravel()):

        img = wrong_images[i]
        img = img * 0.5 + 0.5
        img = img.permute(
            1,
            2,
            0
        )

        ax.imshow(img)

        ax.set_title(
            f"T:{full_dataset.classes[wrong_true[i]]}\n"
            f"P:{full_dataset.classes[wrong_pred[i]]}"
        )

        ax.axis("off")

    plt.tight_layout()

    plt.show()

主程序

In [ ]:
RUN_TRAIN = False